1. Loading the data

In [1]:
import pandas as pd

In [3]:
# Load the large file ----optional
file_path = 'Traffic.csv'
df = pd.read_csv(file_path)

# Split into chunks of 90,000 rows (adjust based on your row size)
chunk_size = 90000
for i, chunk in enumerate(range(0, len(df), chunk_size)):
    chunk_df = df.iloc[chunk:chunk + chunk_size]
    chunk_df.to_csv(f'split_file_{i}.csv', index=False)

/var/folders/7d/zb3rfqvj4rscl8ztf4tzws680000gn/T/ipykernel_34941/141550186.py:3: DtypeWarning: Columns (0: LANE_CNT) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [4]:
df.shape

(1048158, 48)

2. Clean Data

In [5]:
cols_to_drop = ['CRASH_DATE_EST_I']
df = df.drop(columns=cols_to_drop)
df = df.dropna(subset=['LATITUDE'])
df['LOCATION'] = df['LATITUDE'].astype(str) + ',' + df['LONGITUDE'].astype(str)

df['SEVERITY'] = df['MOST_SEVERE_INJURY']

df['SEVERITY'] = df['SEVERITY'].str.replace('NO INDICATION OF INJURY','LOW', case=False)
df['SEVERITY'] = df['SEVERITY'].str.replace('REPORTED, NOT EVIDENT','LOW', case=False)
df['SEVERITY'] = df['SEVERITY'].str.replace('NONINCAPACITATING INJURY','MEDIUM', case=False)
df['SEVERITY'] = df['SEVERITY'].str.replace('INCAPACITATING INJURY','HIGH', case=False)
df['SEVERITY'] = df['SEVERITY'].str.replace('FATAL','HIGH', case=False)

In [6]:
df = df[['CRASH_DATE',
'TRAFFIC_CONTROL_DEVICE',
'WEATHER_CONDITION',
'LIGHTING_CONDITION',
'FIRST_CRASH_TYPE',
'TRAFFICWAY_TYPE',
'ALIGNMENT',
'ROADWAY_SURFACE_COND',
'ROAD_DEFECT',
'CRASH_TYPE',
'INTERSECTION_RELATED_I',
'DAMAGE',
'PRIM_CONTRIBUTORY_CAUSE',
'NUM_UNITS',
'MOST_SEVERE_INJURY',
'INJURIES_TOTAL',
'INJURIES_FATAL',
'CRASH_HOUR',
'CRASH_DAY_OF_WEEK',
'CRASH_MONTH', 'SEVERITY']]

In [7]:
df.to_csv('Traffic_dataset_cleaned.csv', index=False)

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [9]:
# 1. Load Data
df = pd.read_csv('Traffic_dataset_cleaned.csv')

# 0. Remove CRASH_DATE and MOST_SEVERE_INJURY (Data Leakage Fix)
# We MUST remove MOST_SEVERE_INJURY because it is essentially the target.
cols_to_drop = ['CRASH_DATE', 'MOST_SEVERE_INJURY']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Identify Feature Types
target = 'SEVERITY'
numerical_cols = ['NUM_UNITS', 'INJURIES_TOTAL', 'INJURIES_FATAL', 'CRASH_HOUR',
                  'CRASH_DAY_OF_WEEK', 'CRASH_MONTH']
# Filter to columns that actually exist in the CSV
numerical_cols = [c for c in numerical_cols if c in df.columns]
categorical_cols = [col for col in df.columns if col != target and col not in numerical_cols]

print(f'Columns Used: {numerical_cols} \n {categorical_cols}')

# 2. Handling Missing Values (Before splitting to allow stratification)
# Numerical -> Median
for col in numerical_cols:
    if df[col].isnull().all():
        df[col] = df[col].fillna(0)
    else:
        df[col] = df[col].fillna(df[col].median())

# Categorical -> Most Frequent
for col in categorical_cols + [target]:
    if not df[col].isnull().all():
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna('UNKNOWN')

# 5. Data Splitting: 70% Train, 20% Val, 10% Test
try:
    # Attempt Stratified Split
    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df[target])
    val_df, test_df = train_test_split(temp_df, test_size=1/3, random_state=42, stratify=temp_df[target])
except ValueError:
    # Fallback if sample size is too small (like the 5-row sample)
    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Separate Features
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_test, y_test = test_df.drop(columns=[target]), test_df[target]

# 3 & 4. Encoding and Normalization
scaler = MinMaxScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Transform Data
X_train_num = scaler.fit_transform(X_train[numerical_cols])
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_train_final = np.hstack([X_train_num, X_train_cat])

X_test_num = scaler.transform(X_test[numerical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])
X_test_final = np.hstack([X_test_num, X_test_cat])

# 6. Train Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_final, y_train)

# 7. Final Evaluation
y_pred = model.predict(X_test_final)

print(f"--- Performance Metrics: Random Forest (Test Set) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, average='macro'):.4f}")

Columns Used: ['NUM_UNITS', 'INJURIES_TOTAL', 'INJURIES_FATAL', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH'] 
 ['TRAFFIC_CONTROL_DEVICE', 'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE', 'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'CRASH_TYPE', 'INTERSECTION_RELATED_I', 'DAMAGE', 'PRIM_CONTRIBUTORY_CAUSE']
--- Performance Metrics: Random Forest (Test Set) ---
Accuracy: 0.9356
F1 Score: 0.5967
Precision: 0.6602
Recall: 0.6083


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# 1. Load and Clean Dataset
df = pd.read_csv('Traffic_dataset_cleaned.csv')

# 0. Remove irrelevant columns and leakage
# Dropping 'INTERSECTION_RELATED_I' per request and 'MOST_SEVERE_INJURY' to prevent leakage
cols_to_drop = ['CRASH_DATE', 'MOST_SEVERE_INJURY', 'INTERSECTION_RELATED_I']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# 1. Define Column Types
target = 'SEVERITY'
num_cols = ['NUM_UNITS', 'INJURIES_TOTAL', 'INJURIES_FATAL', 'CRASH_HOUR',
            'CRASH_DAY_OF_WEEK', 'CRASH_MONTH']
cat_cols = [c for c in df.columns if c != target and c not in num_cols]


# 2. Handling Missing Values
# Fill numerical with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median() if not df[col].isnull().all() else 0)

# Fill categorical with most frequent
for col in cat_cols + [target]:
    df[col] = df[col].fillna(df[col].mode()[0] if not df[col].isnull().all() else 'UNKNOWN')


# 3. Undersampling LOW severity since they form majority of the dataset
# Separate classes
df_low = df[df['SEVERITY'] == 'LOW']
df_med = df[df['SEVERITY'] == 'MEDIUM']
df_high = df[df['SEVERITY'] == 'HIGH']

#====================================================================
# Sample LOW to 200k rows (adjust n based HIGH/MEDIUM counts)
df_low_under = df_low.sample(n=min(200000, len(df_low)), random_state=42)

# Recombine and shuffle
df_balanced = pd.concat([df_low_under, df_med, df_high], axis=0)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


# 4. Data Splitting (70/20/10 Stratified)
# Step 1: 70% Train, 30% Temp
train_df, temp_df = train_test_split(
    df_balanced,
    test_size=0.30,
    random_state=42,
    stratify=df_balanced[target]
)

# Step 2: Split 30% Temp into 20% Val and 10% Test (1/3 of 30% is 10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.3333,
    random_state=42,
    stratify=temp_df[target]
)


# 5. Feature Scaling and Encoding
scaler = MinMaxScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

def transform_features(data, is_train=False):
    if is_train:
        num_part = scaler.fit_transform(data[num_cols])
        cat_part = encoder.fit_transform(data[cat_cols])
    else:
        num_part = scaler.transform(data[num_cols])
        cat_part = encoder.transform(data[cat_cols])
    return np.hstack([num_part, cat_part])

X_train = transform_features(train_df, is_train=True)
X_test = transform_features(test_df)
y_train = train_df[target]
y_test = test_df[target]


# 6. Train Random Forest
# Added class_weight='balanced' to further help with the 1% High Severity class
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)


# 7. Evaluate Performance
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (Macro): {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (Macro): {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (Macro): {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")

Accuracy: 0.9042
Precision (Macro): 0.7512
Recall (Macro): 0.6671
F1 Score (Macro): 0.6514


In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


# 1. Load and Clean Dataset
df = pd.read_csv('Traffic_dataset_cleaned.csv')

# Remove irrelevant columns and leakage
cols_to_drop = ['CRASH_DATE', 'MOST_SEVERE_INJURY', 'INTERSECTION_RELATED_I']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


# 2. Define Column Types
target = 'SEVERITY'

num_cols = [
    'NUM_UNITS',
    'INJURIES_TOTAL',
    'INJURIES_FATAL',
    'CRASH_HOUR',
    'CRASH_DAY_OF_WEEK',
    'CRASH_MONTH'
]

cat_cols = [c for c in df.columns if c != target and c not in num_cols]


# 3. Handling Missing Values
for col in num_cols:
    df[col] = df[col].fillna(df[col].median() if not df[col].isnull().all() else 0)

for col in cat_cols + [target]:
    df[col] = df[col].fillna(df[col].mode()[0] if not df[col].isnull().all() else 'UNKNOWN')


# 4. Undersampling LOW Severity Class
df_low = df[df[target] == 'LOW']
df_med = df[df[target] == 'MEDIUM']
df_high = df[df[target] == 'HIGH']

df_low_under = df_low.sample(n=min(200000, len(df_low)), random_state=42)

df_balanced = pd.concat([df_low_under, df_med, df_high], axis=0)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


# 5. Data Splitting: 70% Train, 20% Validation, 10% Test
train_df, temp_df = train_test_split(
    df_balanced,
    test_size=0.30,
    random_state=42,
    stratify=df_balanced[target]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.3333,
    random_state=42,
    stratify=temp_df[target]
)


# 6. Feature Scaling and Encoding
scaler = MinMaxScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

def transform_features(data, is_train=False):
    if is_train:
        num_part = scaler.fit_transform(data[num_cols])
        cat_part = encoder.fit_transform(data[cat_cols])
    else:
        num_part = scaler.transform(data[num_cols])
        cat_part = encoder.transform(data[cat_cols])

    return np.hstack([num_part, cat_part])


X_train = transform_features(train_df, is_train=True)
X_val = transform_features(val_df)
X_test = transform_features(test_df)

y_train = train_df[target]
y_val = val_df[target]
y_test = test_df[target]


# 7. Train Logistic Regression Model
log_reg_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)

log_reg_model.fit(X_train, y_train)


# 8. Validation Performance
y_val_pred = log_reg_model.predict(X_val)

print("Validation Results")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision (Macro): {precision_score(y_val, y_val_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (Macro): {recall_score(y_val, y_val_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (Macro): {f1_score(y_val, y_val_pred, average='macro', zero_division=0):.4f}")

print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred, zero_division=0))


# 9. Test Performance
y_test_pred = log_reg_model.predict(X_test)

print("\nTest Results")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision (Macro): {precision_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (Macro): {recall_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (Macro): {f1_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")

print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'

In [15]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


# Encode target labels
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)


# Train XGBoost Model
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train_encoded)


# Predict
y_pred_encoded = xgb_model.predict(X_test)

# Convert predictions back to original labels
y_pred = label_encoder.inverse_transform(y_pred_encoded)


# Evaluate Performance
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (Macro): {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (Macro): {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (Macro): {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.9084
Precision (Macro): 0.8955
Recall (Macro): 0.6675
F1 Score (Macro): 0.6456

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.94      0.06      0.11      1793
         LOW       1.00      0.95      0.97     19999
      MEDIUM       0.75      1.00      0.86      8310

    accuracy                           0.91     30102
   macro avg       0.90      0.67      0.65     30102
weighted avg       0.93      0.91      0.89     30102



In [16]:
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


# Encode target labels
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)


# Train LightGBM Model
lgbm_model = LGBMClassifier(
    objective='multiclass',
    num_class=len(label_encoder.classes_),
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(X_train, y_train_encoded)


# Predict
y_pred_encoded = lgbm_model.predict(X_test)

# Convert predictions back to original labels
y_pred = label_encoder.inverse_transform(y_pred_encoded)


# Evaluate Performance
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (Macro): {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (Macro): {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (Macro): {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019629 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 349
[LightGBM] [Info] Number of data points in the train set: 210730, number of used features: 142
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


/Users/ngwena/PyCharmMiscProject/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.8530
Precision (Macro): 0.6699
Recall (Macro): 0.7100
F1 Score (Macro): 0.6774

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.25      0.47      0.33      1793
         LOW       1.00      0.95      0.97     19999
      MEDIUM       0.76      0.71      0.73      8310

    accuracy                           0.85     30102
   macro avg       0.67      0.71      0.68     30102
weighted avg       0.89      0.85      0.87     30102



In [1]:
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


# Encode target labels
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)


# Compute class weights for imbalanced classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)

class_weights = {
    i: weight for i, weight in enumerate(class_weights_array)
}


# Build MLP Deep Neural Network
mlp_model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(num_classes, activation='softmax')
])


# Compile model
mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


# Train model
history = mlp_model.fit(
    X_train,
    y_train_encoded,
    validation_split=0.2,
    epochs=50,
    batch_size=512,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)


# Predict probabilities
y_pred_probs = mlp_model.predict(X_test)

# Convert probabilities to class labels
y_pred_encoded = np.argmax(y_pred_probs, axis=1)

# Convert predictions back to original labels
y_pred = label_encoder.inverse_transform(y_pred_encoded)


# Evaluate Performance
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (Macro): {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (Macro): {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (Macro): {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

ModuleNotFoundError: No module named 'tensorflow'

In [23]:
!pip install tensorflow
